In [ ]:
%%capture cap
%run ./src/desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

In [ ]:
import earthkit.data

In [ ]:
request = {
     "activity": "story-nudging",
     "class": "d1",
     "dataset": "climate-dt",
     "experiment": "cont",
     "generation": "1",
     "levtype": "sfc",
     "date": "20200102/to/20231231",
     "model": "ifs-fesom",
     "expver": "0001",
     "param": "141",
     "realization": "1",
     "resolution": "high",
     "stream": "clte",
     "type": "fc",
     "time": "1000",
     "feature": {
        #  "type": "timeseries",
        #  "points": [[38, -9.5]],
        #  "time_axis": "date",
        "type": "boundingbox",
        "points" : [[50, 4], [43, 18]], #50°N / 4°E – 43°N / 18°E.
     }
 }

In [ ]:
data = earthkit.data.from_source("polytope", 
    "destination-earth", 
    request, 
    address="polytope.lumi.apps.dte.destination-earth.eu", 
    stream=False)
data_file = "./climate-dt-earthkit-fe-story-nudging.covjson"
data.to_target("file", data_file)

In [ ]:
data_file = "./climate-dt-earthkit-fe-story-nudging.covjson"
data = earthkit.data.from_source("file", data_file) 
dataxr = data.to_xarray()

In [ ]:
print(f"min lon: {dataxr["longitude"].min().values.item()}")
print(f"max lon: {dataxr["longitude"].max().values.item()}")
print(f"min lat: {dataxr["latitude"].min().values.item()}")
print(f"max lat: {dataxr["latitude"].max().values.item()}")
##50°N / 4°E – 43°N / 18°E.

In [ ]:
import xarray as xr
data_file = "./climate-dt-earthkit-fe-story-nudging1.covjson"
data = earthkit.data.from_source("file", data_file) 
snow1 = data.to_xarray()

data_file = "./climate-dt-earthkit-fe-story-nudging.covjson"
data = earthkit.data.from_source("file", data_file) 
snow2 = data.to_xarray()

In [ ]:
snow_xr = xr.concat([snow1, snow2], dim="datetimes")

In [ ]:
snow_xr

In [ ]:
#snow_xr = snow_xr.drop_vars(["levelist", "number", "steps"])
snow_xr = snow_xr.squeeze(["number", "steps"], drop=True)

In [ ]:
snow_xr

In [ ]:
for name in snow_xr.data_vars:
    snow_xr[name] = snow_xr[name].chunk({"datetimes": -1, "points": 1000})
snow_xr.drop_encoding().to_zarr("/Users/christophreimer/datapool/scratch/IFS-FESMO-cont-rechunked.zarr")

In [ ]:
test = xr.open_zarr("/Users/christophreimer/datapool/scratch/IFS-FESMO-cont-rechunked.zarr")
test

In [ ]:
test["sd"]